# Исследование рынка видеоигр 2000-2013 

- Автор: Хабибулина Юлия
- Дата: 28.02.25

### Цели и задачи проекта

Изучить рынок игр, сделать обзор игровых платформ, изучить объёмы продаж игр разных жанров и предпочтения игроков по регионам.

### Описание данных

Даны название игр, название платформы, год выпуска игры, жанр игры, продажи в Северной Америке, Европе, Японии и в других странах, оценка критиков и пользователей, рейтинг организации ESRB.

### Содержимое проекта

1. Загрузка и ознакомление с данными

2. Предобработка- изменение типов данных, очистка от пропусков и дубликатов

3. Категоризация игр по оценкам

4. Выделение самых попудярных платформ

## 1. Загрузка данных и знакомство с ними

In [1]:
import pandas as pd

df = pd.read_csv('/datasets/new_games.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/datasets/new_games.csv'

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


В файле содержатся данные о 17 тысячах игр, но представлены не все данные для каждой игры. Оценки критиков есть только у половины игр, а оценка пользователей и рейтинг- у 10 тысяч. Данные о продажи в Европе и Японии, а так же оценки пользователей и рейтинг представлены не числами.

А еще названия столбцов написаны в не очень удобном формате, далее приведем их к snake_case.

---

## 2.  Проверка ошибок в данных и их предобработка


### 2.1. Названия, или метки, столбцов датафрейма


Названия столбцов написаны не очень удобно- привожу их к snake style. 

In [4]:
print(df.columns)

Index(['Name', 'Platform', 'Year of Release', 'Genre', 'NA sales', 'EU sales',
       'JP sales', 'Other sales', 'Critic Score', 'User Score', 'Rating'],
      dtype='object')


In [5]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
print(df.columns)

Index(['name', 'platform', 'year_of_release', 'genre', 'na_sales', 'eu_sales',
       'jp_sales', 'other_sales', 'critic_score', 'user_score', 'rating'],
      dtype='object')


### 2.2. Типы данных

Данные о продажах в Европе и Японии, а так же оценки пользователей представлены не числами, хотя по описанию данных это числа. Вероятно, это произошло из-за того, что пропуски неправильно заполнили словами, вместо NaN. Привожу их к типу float. В данных содержатся пропуски, но пока не буду их удалять.

In [6]:
df['eu_sales'] = pd.to_numeric(df['eu_sales'], errors='coerce')
df['jp_sales'] = pd.to_numeric(df['jp_sales'], errors='coerce')
df['user_score'] = pd.to_numeric(df['user_score'], errors='coerce')
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float64
 5   eu_sales         16950 non-null  float64
 6   jp_sales         16952 non-null  float64
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       7688 non-null   float64
 10  rating           10085 non-null  object 
dtypes: float64(7), object(4)
memory usage: 1.4+ MB
None


Так же год выпуска представлен не очень удобно- типом float, а не int. Привожу к int (чтобы нули в конце дальше не машались):

In [7]:
df['year_of_release'] = pd.to_numeric(df['year_of_release'], errors='coerce').fillna(0).astype(int)

In [8]:
print(df.head(10))

                        name platform  year_of_release         genre  \
0                 Wii Sports      Wii             2006        Sports   
1          Super Mario Bros.      NES             1985      Platform   
2             Mario Kart Wii      Wii             2008        Racing   
3          Wii Sports Resort      Wii             2009        Sports   
4   Pokemon Red/Pokemon Blue       GB             1996  Role-Playing   
5                     Tetris       GB             1989        Puzzle   
6      New Super Mario Bros.       DS             2006      Platform   
7                   Wii Play      Wii             2006          Misc   
8  New Super Mario Bros. Wii      Wii             2009      Platform   
9                  Duck Hunt      NES             1984       Shooter   

   na_sales  eu_sales  jp_sales  other_sales  critic_score  user_score rating  
0     41.36     28.96      3.77         8.45          76.0         8.0      E  
1     29.08      3.58      6.81         0.77   

### 2.3. Наличие пропусков в данных

Считаю число пропусков в данных- в абсолютном значении и ву процентах от количества строк. 


In [9]:
missing_abs = df.isna().sum(axis=0)
missing_fraction = df.isna().sum(axis=0) / df.shape[0] * 100

print("Абсолютное число пропусков")
print(missing_abs)
print("")
print("Процент пропусков:")
print(missing_fraction)

Абсолютное число пропусков
name                  2
platform              0
year_of_release       0
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating             6871
dtype: int64

Процент пропусков:
name                0.011795
platform            0.000000
year_of_release     0.000000
genre               0.011795
na_sales            0.000000
eu_sales            0.035386
jp_sales            0.023590
other_sales         0.000000
critic_score       51.391838
user_score         54.659118
rating             40.522529
dtype: float64


Пропуски встречаются в столбцах с данными об оценках критиков, оценках пользователей и рейтинге. Небольшое количество пропусков в названиях игр, жанрах и количестве продаж в Европе и Японии. 

В столбцах с оценками пропусков слишком много, чтобы их удалять. Скорее всего, это MCAR пропуски- не все разработчики проводят опросы у пользователей и критиков. То же касается и рейтинга- не все игры проходят валидацию в ESRB. Так как данные важны для дальнейшей работы, попробуем заменить их на средние значения.

In [10]:
mean_sales_eu = df.groupby(['year_of_release', 'platform'])['eu_sales'].transform('mean')
df['eu_sales'] = df['eu_sales'].fillna(mean_sales_eu)

mean_sales_jp = df.groupby(['year_of_release', 'platform'])['jp_sales'].transform('mean')
df['jp_sales'] = df['jp_sales'].fillna(mean_sales_jp)

mean_critic = df.groupby(['year_of_release', 'platform'])['critic_score'].transform('mean')
df['critic_score'] = df['critic_score'].fillna(mean_critic)

mean_user = df.groupby(['year_of_release', 'platform'])['user_score'].transform('mean')
df['user_score'] = df['user_score'].fillna(mean_user)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16956 non-null  int64  
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float64
 5   eu_sales         16956 non-null  float64
 6   jp_sales         16956 non-null  float64
 7   other_sales      16956 non-null  float64
 8   critic_score     15455 non-null  float64
 9   user_score       15679 non-null  float64
 10  rating           10085 non-null  object 
dtypes: float64(6), int64(1), object(4)
memory usage: 1.4+ MB


По количеству оставшихся пустых значений кажется, что получились группы по году платформе, в которых все значения пустые. Так как таких значений больше тысячи и просто удалить их нельзя, то заменю на глобальное среднее.

In [14]:
df['critic_score'] = df['critic_score'].fillna(df['critic_score'].mean())
df['user_score'] = df['user_score'].fillna(df['user_score'].mean())
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16954 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16954 non-null  object 
 2   year_of_release  16954 non-null  int64  
 3   genre            16954 non-null  object 
 4   na_sales         16954 non-null  float64
 5   eu_sales         16954 non-null  float64
 6   jp_sales         16954 non-null  float64
 7   other_sales      16954 non-null  float64
 8   critic_score     16954 non-null  float64
 9   user_score       16954 non-null  float64
 10  rating           16954 non-null  object 
dtypes: float64(6), int64(1), object(4)
memory usage: 1.6+ MB


С рейтингом чуть проще- заменю пропуски на "No", что будет говорить о том, что игра рейтинг не получила. 

In [15]:
df['rating'] = df['rating'].fillna('No')
print(df.head(10))

                        name platform  year_of_release         genre  \
0                 Wii Sports      Wii             2006        Sports   
1          Super Mario Bros.      NES             1985      Platform   
2             Mario Kart Wii      Wii             2008        Racing   
3          Wii Sports Resort      Wii             2009        Sports   
4   Pokemon Red/Pokemon Blue       GB             1996  Role-Playing   
5                     Tetris       GB             1989        Puzzle   
6      New Super Mario Bros.       DS             2006      Platform   
7                   Wii Play      Wii             2006          Misc   
8  New Super Mario Bros. Wii      Wii             2009      Platform   
9                  Duck Hunt      NES             1984       Shooter   

   na_sales  eu_sales  jp_sales  other_sales  critic_score  user_score rating  
0     41.36     28.96      3.77         8.45     76.000000    8.000000      E  
1     29.08      3.58      6.81         0.77   

Осталось разобраться с пропусками в названиях, жанрах и количестве продаж. Их мало (меньше 1%), поэтому просто удаляю. 

In [16]:
df = df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16954 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16954 non-null  object 
 2   year_of_release  16954 non-null  int64  
 3   genre            16954 non-null  object 
 4   na_sales         16954 non-null  float64
 5   eu_sales         16954 non-null  float64
 6   jp_sales         16954 non-null  float64
 7   other_sales      16954 non-null  float64
 8   critic_score     16954 non-null  float64
 9   user_score       16954 non-null  float64
 10  rating           16954 non-null  object 
dtypes: float64(6), int64(1), object(4)
memory usage: 1.6+ MB


### 2.4. Явные и неявные дубликаты

Проверяю на наличие (количество уникальных строк) неявных дубликатов. Проверяю по столбцам с названиями, годом выпуска, платформе и рейтингу- игра могла выйти под одним названием на разных платформах, в разные годы в нескольких частых или в детской и взрослой версии.

In [17]:
num = df[['name', 'year_of_release', 'platform', 'rating']].nunique()
print(num)

name               11559
year_of_release       38
platform              31
rating                 9
dtype: int64


Чтобы убедиться, что все это действительно дубликаты, нормализую данные. Пусть все названия будут маленькими буквами, рейтинг- большими. 

In [18]:
df['name'] = df['name'].str.lower()
df['rating'] = df['rating'].str.upper()
print(df.head(10))

                        name platform  year_of_release         genre  \
0                 wii sports      Wii             2006        Sports   
1          super mario bros.      NES             1985      Platform   
2             mario kart wii      Wii             2008        Racing   
3          wii sports resort      Wii             2009        Sports   
4   pokemon red/pokemon blue       GB             1996  Role-Playing   
5                     tetris       GB             1989        Puzzle   
6      new super mario bros.       DS             2006      Platform   
7                   wii play      Wii             2006          Misc   
8  new super mario bros. wii      Wii             2009      Platform   
9                  duck hunt      NES             1984       Shooter   

   na_sales  eu_sales  jp_sales  other_sales  critic_score  user_score rating  
0     41.36     28.96      3.77         8.45     76.000000    8.000000      E  
1     29.08      3.58      6.81         0.77   

Оцениваю дубликаты:

In [19]:
duplicates = df[df.duplicated(keep=False)]
print(duplicates.head(10))

                                 name platform  year_of_release         genre  \
267             batman: arkham asylum      PS3             2009        Action   
268             batman: arkham asylum      PS3             2009        Action   
367  james bond 007: agent under fire      PS2             2001       Shooter   
368  james bond 007: agent under fire      PS2             2001       Shooter   
716             god of war: ascension      PS3             2013        Action   
717             god of war: ascension      PS3             2013        Action   
847   rayman raving rabbids: tv party      Wii             2008          Misc   
848   rayman raving rabbids: tv party      Wii             2008          Misc   
962                        diablo iii      PS4             2014  Role-Playing   
963                        diablo iii      PS4             2014  Role-Playing   

     na_sales  eu_sales  jp_sales  other_sales  critic_score  user_score  \
267      2.24      1.31      0.0

Действительно, дубликаты есть, удалим все строки, кроме первой найденной.

In [20]:
df = df.drop_duplicates(keep='first')
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16772 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16772 non-null  object 
 1   platform         16772 non-null  object 
 2   year_of_release  16772 non-null  int64  
 3   genre            16772 non-null  object 
 4   na_sales         16772 non-null  float64
 5   eu_sales         16772 non-null  float64
 6   jp_sales         16772 non-null  float64
 7   other_sales      16772 non-null  float64
 8   critic_score     16772 non-null  float64
 9   user_score       16772 non-null  float64
 10  rating           16772 non-null  object 
dtypes: float64(6), int64(1), object(4)
memory usage: 1.5+ MB


Посчиатем, сколько строк было утеряно в бою...

In [21]:
print(f"Удалено строк: {16956 - 16762}")
pr = round(194*100/16956, 2)
print(f"Процент удаленных строк: {pr}%")

Удалено строк: 194
Процент удаленных строк: 1.14%


После предобработки у нас осталось 99% строк без пропусков, дубликатов и прочей нечести. 

---

## 3. Фильтрация данных

Создадим датафрейм с данными об играх, выпущенных с 2000 по 2013 год. 

In [23]:
df_actual = df.loc[(df['year_of_release'] >= 2000) & (df['year_of_release'] <= 2013)]
print(df_actual.head(10))

                         name platform  year_of_release       genre  na_sales  \
0                  wii sports      Wii             2006      Sports     41.36   
2              mario kart wii      Wii             2008      Racing     15.68   
3           wii sports resort      Wii             2009      Sports     15.61   
6       new super mario bros.       DS             2006    Platform     11.28   
7                    wii play      Wii             2006        Misc     13.96   
8   new super mario bros. wii      Wii             2009    Platform     14.44   
10                 nintendogs       DS             2005  Simulation      9.05   
11              mario kart ds       DS             2005      Racing      9.71   
13                    wii fit      Wii             2007      Sports      8.92   
14         kinect adventures!     X360             2010        Misc     15.00   

    eu_sales  jp_sales  other_sales  critic_score  user_score rating  
0      28.96      3.77         8.45  

---

## 4. Категоризация данных
    
Разделим игры по оценкам пользвателей. Будем считать, что если игру оценили меньше, чем на 3 балла- оценка низкая, до 8 баллов- средняя, больше 8- высокая.

In [25]:
import warnings
warnings.filterwarnings('ignore')
df_actual['category_user'] = pd.cut(df_actual['user_score'], bins=[0, 3, 8, 10], labels=["low", "middle", "high"], right=False)
print(df_actual.head(10))

                         name platform  year_of_release       genre  na_sales  \
0                  wii sports      Wii             2006      Sports     41.36   
2              mario kart wii      Wii             2008      Racing     15.68   
3           wii sports resort      Wii             2009      Sports     15.61   
6       new super mario bros.       DS             2006    Platform     11.28   
7                    wii play      Wii             2006        Misc     13.96   
8   new super mario bros. wii      Wii             2009    Platform     14.44   
10                 nintendogs       DS             2005  Simulation      9.05   
11              mario kart ds       DS             2005      Racing      9.71   
13                    wii fit      Wii             2007      Sports      8.92   
14         kinect adventures!     X360             2010        Misc     15.00   

    eu_sales  jp_sales  other_sales  critic_score  user_score rating  \
0      28.96      3.77         8.45 

Разделим игры по оценкам критиков. Будем считать, что если игру оценили меньше, чем на 30 баллов- оценка низкая, до 80 баллов- средняя, больше 80- высокая.

In [27]:
warnings.filterwarnings('ignore')
df_actual['category_critic'] = pd.cut(df_actual['critic_score'], bins=[0, 30, 80, 100], labels=["low", "middle", "high"], right=False)
print(df_actual.head(10))

                         name platform  year_of_release       genre  na_sales  \
0                  wii sports      Wii             2006      Sports     41.36   
2              mario kart wii      Wii             2008      Racing     15.68   
3           wii sports resort      Wii             2009      Sports     15.61   
6       new super mario bros.       DS             2006    Platform     11.28   
7                    wii play      Wii             2006        Misc     13.96   
8   new super mario bros. wii      Wii             2009    Platform     14.44   
10                 nintendogs       DS             2005  Simulation      9.05   
11              mario kart ds       DS             2005      Racing      9.71   
13                    wii fit      Wii             2007      Sports      8.92   
14         kinect adventures!     X360             2010        Misc     15.00   

    eu_sales  jp_sales  other_sales  critic_score  user_score rating  \
0      28.96      3.77         8.45 

Группируем данные по категориям и считаем, сколько игр попало в каждую. Общая, по оценкам пользователей и по оценкам критиков. 

In [28]:
by_groups = df_actual.groupby(['category_user', 'category_critic']).count()
print(by_groups['platform'])

category_user  category_critic
low            low                  17
               middle               98
               high                  1
middle         low                  37
               middle             9633
               high                681
high           low                   1
               middle             1279
               high               1082
Name: platform, dtype: int64


In [29]:
by_groups = df_actual.groupby('category_user').count()
print(by_groups['platform'])

category_user
low         116
middle    10351
high       2362
Name: platform, dtype: int64


In [30]:
by_groups = df_actual.groupby('category_critic').count()
print(by_groups['platform'])

category_critic
low          55
middle    11010
high       1764
Name: platform, dtype: int64


In [31]:
max_score_u = df_actual['user_score'].max()
print(f'Максимальная оценка пользователей: {max_score_u}')
user_best = df_actual.loc[df['user_score'] == 9.7, ['name', 'genre']]
print("Игра с высшей оценкой пользователей")
print(user_best)

Максимальная оценка пользователей: 9.7
Игра с высшей оценкой пользователей
                     name         genre
14611  breath of fire iii  Role-Playing


In [32]:
max_score_c = df_actual['critic_score'].max()
print(f'Максимальная оценка пользователей: {max_score_c}')
critic_best = df_actual.loc[df['critic_score'] == 98.0, ['name', 'genre']]
print("Игры с высшей оценкой критиков")
print(critic_best)

Максимальная оценка пользователей: 98.0
Игры с высшей оценкой критиков
                         name   genre
51        grand theft auto iv  Action
57        grand theft auto iv  Action
227  tony hawk's pro skater 2  Sports


Выделим топ-7 платформ, на которых выходило больше всего игр с 2000 по 2013 годы. 

In [33]:
platform_counts = df_actual['platform'].value_counts()
top_7_platforms = platform_counts.nlargest(7)
print(top_7_platforms)

PS2     2131
DS      2126
Wii     1279
PSP     1184
X360    1126
PS3     1090
GBA      813
Name: platform, dtype: int64


## 5. Исследование по жанру и регионам

Заказчики хотят сделать акцент на играх в жанре RPG, поэтому извлечем данные только о них:

In [34]:
df_rpg = df_actual.loc[(df_actual['genre'] == 'Role-Playing')]
df_rpg.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1079 entries, 20 to 16942
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   name             1079 non-null   object  
 1   platform         1079 non-null   object  
 2   year_of_release  1079 non-null   int64   
 3   genre            1079 non-null   object  
 4   na_sales         1079 non-null   float64 
 5   eu_sales         1079 non-null   float64 
 6   jp_sales         1079 non-null   float64 
 7   other_sales      1079 non-null   float64 
 8   critic_score     1079 non-null   float64 
 9   user_score       1079 non-null   float64 
 10  rating           1079 non-null   object  
 11  category_user    1079 non-null   category
 12  category_critic  1079 non-null   category
dtypes: category(2), float64(6), int64(1), object(4)
memory usage: 103.5+ KB


Теперь сгруппируем данные по количеству продаж в разных регионах:

In [35]:
max_na = df_rpg['na_sales'].max()
print(f'Максимальное количество продаж в Америке: {max_na}')
max_eu = df_rpg['eu_sales'].max()
print(f'Максимальное количество продаж в Европе: {max_eu}')
max_jp = df_rpg['jp_sales'].max()
print(f'Максимальное количество продаж в Японии: {max_eu}')

Максимальное количество продаж в Америке: 6.38
Максимальное количество продаж в Европе: 6.21
Максимальное количество продаж в Японии: 6.21


Тогда поделим на три равные группы: если продаж меньше 2 млн, то их мало, до 4 млн- средне, больше 4- много.

In [36]:
warnings.filterwarnings('ignore')
df_rpg['na_category'] = pd.cut(df_rpg['na_sales'], bins=[0, 2, 4, 7], labels=["low sales", "middle sales", "high sales"], right=False)
df_rpg['eu_category'] = pd.cut(df_rpg['eu_sales'], bins=[0, 2, 4, 7], labels=["low sales", "middle sales", "high sales"], right=False)
df_rpg['jp_category'] = pd.cut(df_rpg['jp_sales'], bins=[0, 2, 4, 7], labels=["low sales", "middle sales", "high sales"], right=False)
print(df_rpg.head(5))

                                 name platform  year_of_release         genre  \
20      pokemon diamond/pokemon pearl       DS             2006  Role-Playing   
25      pokemon ruby/pokemon sapphire      GBA             2002  Role-Playing   
27        pokemon black/pokemon white       DS             2010  Role-Playing   
33                pokemon x/pokemon y      3DS             2013  Role-Playing   
58  pokemon firered/pokemon leafgreen      GBA             2004  Role-Playing   

    na_sales  eu_sales  jp_sales  other_sales  critic_score  user_score  \
20      6.38      4.46      6.04         1.36     63.354167    6.912346   
25      6.06      3.90      5.38         0.50     66.854962    7.451786   
27      5.51      3.17      5.65         0.80     67.912281    7.308333   
33      5.28      4.19      4.35         0.78     66.700000    6.620000   
58      4.34      2.65      3.15         0.35     63.940476    7.796429   

   rating category_user category_critic na_category   eu_categ

Сгруппируем по регионам и посчитаем количество продаж:

In [37]:
na_groups = df_rpg.groupby('na_category').count()
print("Продажи в Америке")
print(na_groups['name'])

Продажи в Америке
na_category
low sales       1054
middle sales      19
high sales         6
Name: name, dtype: int64


In [38]:
eu_groups = df_rpg.groupby('eu_category').count()
print("Продажи в Европе")
print(eu_groups['name'])

Продажи в Европе
eu_category
low sales       1068
middle sales       8
high sales         3
Name: name, dtype: int64


In [39]:
jp_groups = df_rpg.groupby('jp_category').count()
print("Продажи в Японии")
print(jp_groups['name'])

Продажи в Японии
jp_category
low sales       1060
middle sales      11
high sales         8
Name: name, dtype: int64


По всем странам игры в среднем разошлись меньше, чем по 2 млн копий. Лучшие показатели в Амкрике- 19 игр продалось в размере до 4 млн копий, но в Японии аж 8 игр купиои более 4 млн раз!

Хочу посмотреть, какая игра лучше всего продавалась в мире:

In [40]:
max_na_g = df_actual['na_sales'].max()
print(f'Максимальное количество продаж в Америке: {max_na_g}')
max_eu_g = df_actual['eu_sales'].max()
print(f'Максимальное количество продаж в Европе: {max_eu_g}')
max_jp_g = df_actual['jp_sales'].max()
print(f'Максимальное количество продаж в Японии: {max_jp_g}')

Максимальное количество продаж в Америке: 41.36
Максимальное количество продаж в Европе: 28.96
Максимальное количество продаж в Японии: 6.5


In [43]:
na_best = df_actual.loc[df['na_sales'] == 41.36, ['name', 'genre']]
print("Самая популярная в Америке")
print(na_best)
eu_best = df_actual.loc[df['eu_sales'] == 28.96, ['name', 'genre']]
print("Самая популярная в Европе")
print(eu_best)
jp_best = df_actual.loc[df['jp_sales'] == 6.5, ['name', 'genre']]
print("Самая популярная в Японии")
print(jp_best)

Самая популярная в Америке
         name   genre
0  wii sports  Sports
Самая популярная в Европе
         name   genre
0  wii sports  Sports
Самая популярная в Японии
                    name     genre
6  new super mario bros.  Platform


In [44]:
max_ot_g = df_actual['other_sales'].max()
print(f'Максимальное количество продаж в мире: {max_ot_g}')
ot_best = df_actual.loc[df['other_sales'] == 10.57, ['name', 'genre']]
print("Самая популярная в мире")
print(ot_best)

Максимальное количество продаж в мире: 10.57
Самая популярная в мире
                             name   genre
17  grand theft auto: san andreas  Action


В Америке и в Европе самой популярной игрой в 2000-2013 годах была Wii Sports, в то время как в Японии отдавали предпочтение Mario. В остальном мире лучше всего продавалась ГТА Сан Андреас. 

---

## 6. Итоговый вывод


В топ платформ вошли, ожидаемо PS, DS и Wii. Игры в жанре RPG имеют не лучшие показатели продаж в целом. 
В среднем игры получали от пользователей и критиков средние оценки (привет, нормальное распределение), но конечно были выбросы- наивысшие оценки пользователей получила RPG игра Breath of Fire III, а критики выразили наибольшую симпатию (внезапно, экшену) ГТА и спортивной игре про скейтинг. 

Самой большой популярностью пользуется игра Wii sports (оправдано, у меня в детстве такая была, просто обожала ее). При этом в Японии самой популярной игрой оказался Марио (тоже ожидаемо, они любят пиксели). В остальном мире самая популярная игра- ГТА (страны бедные и крутые машины только в игре).